In [72]:
import os
import struct
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score
from agcounts_filter import convert_AC
from collections import Counter
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from scipy.signal import medfilt
from scipy.ndimage import maximum_filter1d

In [ ]:
folder_path = "C:/Users/roman/Documents/BEaCHILD/X_et_Y" 
folder = os.listdir(folder_path)


# Initialisation des listes contenant les données des capteurs
X_non_dom = []
X_dom = []
idx_X_non_dom = 0
idx_X_dom = 0
idx_Y_dom = 0
idx_Y_non_dom = 0

# Initialisation des listes contenant les annotations vidéos
Y_non_dom = [] 
Y_dom = []

#Parcourir les fichiers CSV
for file in folder[0:4]: #[0:7]
    print(f"On est dans le fichier : {file}")

    # Extension du fichier
    extension = os.path.splitext(file)[1] 

    # Pour les cas ou il y a des annotations avant le start
    allow_start_dom = None
    allow_start_non_dom = None

    """********************************************************************** 1. Lecture des fichiers csv ******************************************************************************"""
    if extension == ".csv":
        # Récupération des noms des colonnes pour pouvoir lire le fichier (pas le m^me pour tous)
        csv_file_column = pd.read_csv(folder_path + "/" + file, skiprows=5, sep = ',', low_memory = False)
        lst_column = list(csv_file_column)

        # Les fichiers n'ont pas tous le même nombre de données
        if lst_column == ["Time","Gyro X","Gyro Y","Gyro Z","Accel X","Accel Y","Accel Z","Event","Quat W","Quat X","Quat Y","Quat Z","Unnamed: 12"]:
            data_file = pd.read_csv(folder_path + "/" + file, skiprows=7,decimal=",", names = ["Time","Gyro X","Gyro Y","Gyro Z","Accel X","Accel Y","Accel Z","Event","Quat W","Quat X","Quat Y","Quat Z","Unnamed: 12"], dtype = {"Event":str},low_memory=False)
        elif lst_column == ['Time', 'Gyro X', 'Gyro Y', 'Gyro Z', 'Accel X', 'Accel Y', 'Accel Z', 'Quat W', 'Quat X', 'Quat Y', 'Quat Z', 'Unnamed: 11']:
            data_file = pd.read_csv(folder_path + "/" + file, skiprows=7,decimal=",", names = ["Time","Gyro X","Gyro Y","Gyro Z","Accel X","Accel Y","Accel Z","Quat W","Quat X","Quat Y","Quat Z"],low_memory=False) #names=list(dtype_dict.keys()), dtype=dtype_dict)

        # Conversion en AC 
        dom_AC = convert_AC(folder_path + "/" + file)

        # Enregistrement des données des capteurs dans les listes qui seront donnnées au classificateur
        if file[-11:-4] == "non_dom":
            list_dom_AC = dom_AC["AC"].tolist() # Conversion du type pd.serie en type list
            for ac in list_dom_AC :
                X_non_dom.append(ac)

        elif file[-7:-4] == "dom":
            list_dom_AC = dom_AC["AC"].tolist()
            for ac in list_dom_AC:
                X_dom.append(ac)

        

    """********************************************************** 2. Enregistrer les annotations ***************************************************************************"""

    if extension == ".xlsx":

        my_wb = openpyxl.load_workbook(folder_path + "/" + file) 
        my_sheet = my_wb.active

        # Initialisation des décalages : sert pour éviter le décalage du aux arrondis des annotations 
        decalage_dom = 0
        decalage_non_dom = 0

        for label in my_sheet["H"]: 

            """*********************************************** Récupération du début de Y ****************************************************"""
            # dom
            # Synchronisation des données X des capteurs et des annotations Y
            if label.value == "Start_LW" :
                start_dom = int(round(float(my_sheet.cell(label.row,13).value))) - 1
                # Décalage pour synchroniser X et Y
                for decalage in range(start_dom):
                    Y_dom.append(None)
                previous_row_dom = label.row
                # Pour les cas ou il y a des annotations avant le start 
                allow_start_dom = True

            # non_dom 
            # Synchronisation des données X des capteurs et des annotations Y
            if label.value == "Start_RW" :
                start_non_dom = int(round(float(my_sheet.cell(label.row,13).value))) - 1
                # Décalage pour synchroniser X et Y
                for decalage in range(start_non_dom):
                    Y_non_dom.append(None)
                previous_row_non_dom = label.row
                # Pour les cas ou il y a des annotations avant le start 
                allow_start_non_dom = True

            """****************************************************** Récupération du contenu de Y ****************************************"""

            if label.value[0:2] == "LW" and allow_start_dom:

                # Récupération de l'index de la ligne
                row = label.row

                # Synchronisation si le start de l'annotation ne correspond pas au stop de l'annotation précédentes
                if my_sheet.cell(row, 12).value != my_sheet.cell(previous_row_dom, 13).value:
                    difference = round(float(my_sheet.cell(row, 12).value)) - round(float(my_sheet.cell(previous_row_dom , 13).value)) 
                    for diff in range(0, difference):
                        Y_dom.append(None) 
                    
                # Synchronisation des données des capteurs avec les annotations
                nb_repetitions = int(round(float(my_sheet.cell(label.row, 13).value))) - int(round(float(my_sheet.cell(label.row, 12).value))) 
                for nb_sec in range(nb_repetitions): 
                    if X_dom and (len(Y_dom) == len(X_dom)) :
                        break
                    if label.value[-2:] == "GA" or label.value[-2:] == "FA": # GA : # if label.value[-5:] == " V GA": # FAGA : # if label.value[-5:] == " V GA" or label.value[-5:] == " V FA": # ALL : # if label.value[-2:] == "GA" or label.value[-2:] == "FA":
                        Y_dom.append("mouvement")
                    elif label.value[3:13] == "sédentaire": # GA : # elif label.value[3:13] == "sédentaire" or label.value[-5:] == "IV GA" or label.value[-2:] == "FA": # FAGA # elif label.value[3:13] == "sédentaire" or label.value[-5:] == "IV GA" or label.value[-5:] == "IV FA": # ALL : #  elif label.value[3:13] == "sédentaire":
                        Y_dom.append("non mouvement")
                    elif label.value[3:12] == "non noté":
                        Y_dom.append(None) 

                #Au prochain tour, l'indice de la ligne actuelle sera l'index de la ligne précédente
                previous_row_dom = label.row


            elif label.value[0:2] == "RW" and allow_start_non_dom:
                
                # Récupération de l'index de la ligne
                row = label.row

                # Synchronisation si le start de l'annotation ne correspond pas au stop de l'annotation précédentes
                if my_sheet.cell(row, 12).value != my_sheet.cell(previous_row_non_dom, 13).value:
                    difference = round(float(my_sheet.cell(row, 12).value)) - round(float(my_sheet.cell(previous_row_non_dom , 13).value)) 
                    for diff in range(0,difference):
                        Y_non_dom.append(None) #.append("décalage annotations")

                # Synchronisation des données des capteurs avec les annotations
                nb_repetitions = int(round(float(my_sheet.cell(label.row, 13).value))) - int(round(float(my_sheet.cell(label.row, 12).value)))
                for nb_sec in range(nb_repetitions): 
                    if X_non_dom and (len(Y_non_dom) == len(X_non_dom)): 
                        break
                    if label.value[-2:] == "GA" or label.value[-2:] == "FA": # GA : # if label.value[-5:] == " V GA": # FAGA : # if label.value[-5:] == " V GA" or label.value[-5:] == " V FA": # ALL : # if label.value[-2:] == "GA" or label.value[-2:] == "FA":
                        Y_non_dom.append("mouvement")
                    elif label.value[3:13] == "sédentaire": # GA : # elif label.value[3:13] == "sédentaire" or label.value[-5:] == "IV GA" or label.value[-2:] == "FA": # FAGA # elif label.value[3:13] == "sédentaire" or label.value[-5:] == "IV GA" or label.value[-5:] == "IV FA": # ALL : #  elif label.value[3:13] == "sédentaire"
                        Y_non_dom.append("non mouvement")
                    
                    elif label.value[3:12] == "non noté":
                        Y_non_dom.append(None)
                        

                #Au prochain tour, l'indice de la ligne actuelle sera l'index de la ligne précédente
                previous_row_non_dom = label.row

        # CROPER X A LA LONGUEUR DE Y 
        X_dom = X_dom[:len(Y_dom)] 
        X_non_dom = X_non_dom[:len(Y_non_dom)]
        Y_dom = Y_dom[:len(X_dom) ]
        Y_non_dom = Y_non_dom[:len(X_non_dom) ]

Y_dom = np.array(Y_dom)
Y_non_dom = np.array(Y_non_dom)
X_dom = np.array(X_dom)
X_non_dom = np.array(X_non_dom)

total_list_dom_sec = np.arange(1,len(Y_dom)+1)
total_list_non_dom_sec = np.arange(1,len(Y_non_dom)+1)

# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
mask_dom = Y_dom != None
# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
mask_non_dom = Y_non_dom != None

X_dom = X_dom[mask_dom]
X_non_dom = X_non_dom[mask_non_dom]
list_dom_sec =  total_list_dom_sec[mask_dom]
list_non_dom_sec = total_list_non_dom_sec[mask_non_dom]




On est dans le fichier : Autres
On est dans le fichier : Data_10_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_10_non_dom.csv
Reading in CSV
Converting to array
Getting Counts
On est dans le fichier : Data_10_X.xlsx
7191
7191


In [74]:
print(mask_non_dom)
#print(len(X_dom))
#print(len(X_non_dom))
print(list_dom_sec[0:10])
print(list_non_dom_sec[0:10])
print(len(list_dom_sec))

print("")
print(Y_dom)

[False False False ...  True  True  True]
[125 126 127 128 129 130 131 132 133 134]
[125 126 127 128 129 130 131 132 133 134]
2359

[None None None ... 'mouvement' 'mouvement' 'mouvement']


AC > 2

In [75]:
lst_X_members = [X_dom, X_non_dom]
lst_Y_members = [Y_dom, Y_non_dom]
X_members = []
Y_members = []
lst_Y_pred = []

for idx_members in range(2):
    #On a X_dom et Y_dom : construire un Y_pred et le comparer à Y_dom
    Y_pred_2 = []
    X_members.append(lst_X_members[idx_members])
    Y_members.append(lst_Y_members[idx_members])

    # Seuillage : mouvemetn si AC > 2
    for idx_data in range(0,len(X_members[idx_members])):
        seuil = X_members[idx_members][idx_data]
        if seuil > 2 :
            Y_pred_2.append(1)
        elif seuil <= 2 :
            Y_pred_2.append(0)
        else:
            Y_pred_2.append(None)
    
    lst_Y_pred.append(Y_pred_2)

print(len(X_non_dom))
print(len(lst_Y_pred[0]))
print("")

2434
2359



Coley

In [76]:
folder_path = "C:/Users/roman/Documents/BEaCHILD/X_et_Y" 
folder = os.listdir(folder_path)

# Initialisation des listes contenant les données des capteurs
X_gyro_dom = []
Y_gyro_dom = []
Z_gyro_dom = []
X_gyro_non_dom = []
Y_gyro_non_dom = []
Z_gyro_non_dom = []

for file in folder[0:4]: 
    print(f"On est dans le fichier : {file}")

    # Extension du fichier
    extension = os.path.splitext(file)[1] 

    if extension == ".csv":
        # Récupération des données des capteurs
        data_file = pd.read_csv(folder_path + "/" + file, header = 5, names = ["Timestamp","Gyro X","Gyro Y","Gyro Z","Accelerometer X","Accelerometer Y","Accelerometer Z","Event","Quat W","Quat X","Quat Y","Quat Z","None"])    
        
        if file[-11:-4] == "non_dom":
            X_gyro_non_dom.extend(data_file["Gyro X"].values.tolist()) 
            Y_gyro_non_dom.extend(data_file["Gyro Y"].values.tolist()) 
            Z_gyro_non_dom.extend(data_file["Gyro Z"].values.tolist())
        
        elif file[-7:-4] == "dom":
            X_gyro_dom.extend(data_file["Gyro X"].values.tolist()) #.append(data_file["Gyro X"][data])
            Y_gyro_dom.extend(data_file["Gyro Y"].values.tolist()) #.append(data_file["Gyro Y"][data])
            Z_gyro_dom.extend(data_file["Gyro Z"].values.tolist()) #.append(data_file["Gyro Z"][data])

print(len(X_gyro_dom))

On est dans le fichier : Autres
On est dans le fichier : Data_10_dom.csv
On est dans le fichier : Data_10_non_dom.csv
On est dans le fichier : Data_10_X.xlsx
920495


In [77]:

#-------------------------- Extraire les 3 axes gyroscopiques --------------------------
gyro_x = X_gyro_dom
gyro_y = Y_gyro_dom
gyro_z = Z_gyro_dom

#-------------------------- Détection des pics > 10°/s sur chaque axe --------------------------
mask_x = np.abs(np.array(gyro_x)) > 10
mask_y = np.abs(np.array(gyro_y)) > 10
mask_z = np.abs(np.array(gyro_z)) > 10

detected_x = np.abs(np.array(gyro_x)[mask_x])
detected_y = np.abs(np.array(gyro_y)[mask_y])
detected_z = np.abs(np.array(gyro_z)[mask_z])

#-------------------------- Moyenne des pics par axe --------------------------
    
mean_x = detected_x.mean() if len(detected_x) > 0 else np.inf
mean_y = detected_y.mean() if len(detected_y) > 0 else np.inf
mean_z = detected_z.mean() if len(detected_z) > 0 else np.inf

#-------------------------- Seuil adaptatif = min des moyennes --------------------------
adaptive_threshold = min(mean_x, mean_y, mean_z)
print(f"Seuil adaptatif : {adaptive_threshold:.2f} °/s")

#-------------------------- Détection d'un mouvement si un des 3 axes depasse le seuil --------------------------
   
movement_raw = ((np.abs(gyro_x) > adaptive_threshold) | (np.abs(gyro_y) > adaptive_threshold) | (np.abs(gyro_z) > adaptive_threshold)).astype(int)

#-------------------------- Filtre max mobile (fusionner les mouvements séparés de < 0.5s) --------------------------
sampling_rate = 128 
merge_window = int(1 * sampling_rate)  
gap_threshold = int(0.5 * sampling_rate)  
min_duration = int(1.5 * sampling_rate) 

movement_merged = maximum_filter1d(movement_raw, size = merge_window)

#-------------------------- Filtre median mobile (supprimer les mouvements < 1.5s) --------------------------
movement_prediction = medfilt(movement_merged, kernel_size = min_duration | 1)  # 1 car le kernel doit etre impair

#-------------------------- Sous echantilloner pour avoir une annotation par seconde (et non tous les 128 sec) --------------------------
def downsample_to_1Hz(binary_array, sampling_rate=128):
    n_seconds = len(binary_array) // sampling_rate
    binary_array = binary_array[:n_seconds * sampling_rate]  # couper pour etre multiple de 128
    reshaped = binary_array.reshape(n_seconds, sampling_rate) # Majorité : si au moins 64 échantillons sont actifs, on considère la seconde comme mouvement
    return (reshaped.sum(axis=1) >= (sampling_rate // 2)).astype(int)

predicted_1Hz = downsample_to_1Hz(movement_prediction)
# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
predicted_1Hz = predicted_1Hz[:len(Y_dom)] 

mask_dom = Y_dom != None
# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
mask_non_dom = Y_non_dom != None

predicted_1Hz = predicted_1Hz[mask_dom]

print(len(predicted_1Hz))
print(predicted_1Hz)

Seuil adaptatif : 46.87 °/s
2359
[0 0 0 ... 0 1 1]


Enregistrement dans excel pour membre dominant

In [ ]:
file_path_dom = "C:/Users/roman/Documents/BEaCHILD/Classification/Classification/Comparaison_mindmaze/Comparaison_mindmaze_dom_02.06.01_Stage_MS_01.xlsx"
# Select file
if os.path.exists(file_path_dom):
    my_wb = openpyxl.load_workbook(file_path_dom)  
    print("File exists, loading existing data.")
else:
    # Create file 
    my_wb = openpyxl.Workbook()
    print("File does not exist, creating a new file.")

# Nomenclature des colonnes
my_sheet = my_wb.active



names_column = ["Timecode (sec)", "Coley Prediction (0 no movement, 1 movement)", "AC > 2 prediction (0 no movement, 1 movement)", "GA observation (0 no movement, 1 movement)", "GAFA Observation (0 no movement, 1 movement)", "all observation (0 no movement, 1 movement)"]
for name_nb in range(0,len(names_column)) :
    print("créer une colonne")
    my_cell = my_sheet.cell(row = 1, column = name_nb + 1)
    my_cell.value = names_column[name_nb]


# 1ere colonne = seconde
for idx_sec in range(len(list_dom_sec)):
    my_cell = my_sheet.cell(row = idx_sec + 2, column = 1)
    my_cell.value = list_dom_sec[idx_sec]

# 2e colonne = résultats Coley
for idx_pred_coley in range(len(predicted_1Hz)):
    my_cell = my_sheet.cell(row = idx_pred_coley + 2, column = 2)
    my_cell.value = predicted_1Hz[idx_pred_coley]

# 3e colonne = résultats AC > 2
for idx_pred_AC in range(len(lst_Y_pred[0])):
    my_cell = my_sheet.cell(row = idx_pred_AC + 2, column = 3)
    my_cell.value = lst_Y_pred[0][idx_pred_AC]



# Créer un masque pour garder uniquement les étiquettes valides (différent de None)
mask_dom = Y_dom != None
mask_non_dom = Y_non_dom != None
Y_dom = Y_dom[mask_dom]
Y_non_dom = Y_non_dom[mask_non_dom]

"""
# 4e colonne = Observation GA
for idx_obs_ga in range(len(Y_dom)):
    my_cell = my_sheet.cell(row = idx_obs_ga + 2, column = 4)
    if Y_dom[idx_obs_ga] == "mouvement":
        my_cell.value = 1
    elif Y_dom[idx_obs_ga] == "non mouvement":
        my_cell.value = 0
    else:
        my_cell.value = None
"""
"""
# 5e colonne = Observation GAFA
for idx_obs_gafa in range(len(Y_dom)):
    my_cell = my_sheet.cell(row = idx_obs_gafa + 2, column = 5)
    if Y_dom[idx_obs_gafa] == "mouvement":
        my_cell.value = 1
    elif Y_dom[idx_obs_gafa] == "non mouvement":
        my_cell.value = 0
    else:
        print(Y_dom[idx_obs_gafa])
        my_cell.value = "ERREUR"

"""

# 6e colonne = Observation All
for idx_obs_all in range(len(Y_dom)):
    my_cell = my_sheet.cell(row = idx_obs_all + 2, column = 6)
    if Y_dom[idx_obs_all] == "mouvement":
        my_cell.value = 1
    elif Y_dom[idx_obs_all] == "non mouvement":
        my_cell.value = 0
    else:
        print(Y_dom[idx_obs_all])
        my_cell.value = "ERREUR"



my_wb.save(file_path_dom)

File exists, loading existing data.
créer une colonne
créer une colonne
créer une colonne
créer une colonne
créer une colonne
créer une colonne


In [79]:
print(len(Y_non_dom))

2434


Enregistrement dans excel pour membre non dominant

In [80]:
"""file_path_dom = "C:/Users/roman/Documents/BEaCHILD/Classification/Classification/Comparaison_mindmaze_dom_mouv_All.xlsx"
# Select file
if os.path.exists(file_path_dom):
    my_wb = openpyxl.load_workbook(file_path_dom)  
    print("File exists, loading existing data.")
else:
    # Create file 
    my_wb = openpyxl.Workbook()
    print("File does not exist, creating a new file.")

# Nomenclature des colonnes
my_sheet = my_wb.active
names_column = ["Timecode (sec)", "Coley Prediction (0 no movement, 1 movement)", "AC > 2 prediction (0 no movement, 1 movement)"]
for name_nb in len(names_column) :
    my_sheet.cell(row = 1, column = name_nb)
    my_sheet.value = names_column[name_nb]

# 1ere colonne = seconde
for idx_sec in range(len(list_dom_sec)):
    my_sheet.cell(row = idx_sec + 1, column = 0)
    my_sheet.value = list_dom_sec[idx_sec]

# 2e colonne = résultats Coley

# 3e colonne = résultats AC > 2"""

'file_path_dom = "C:/Users/roman/Documents/BEaCHILD/Classification/Classification/Comparaison_mindmaze_dom_mouv_All.xlsx"\n# Select file\nif os.path.exists(file_path_dom):\n    my_wb = openpyxl.load_workbook(file_path_dom)  \n    print("File exists, loading existing data.")\nelse:\n    # Create file \n    my_wb = openpyxl.Workbook()\n    print("File does not exist, creating a new file.")\n\n# Nomenclature des colonnes\nmy_sheet = my_wb.active\nnames_column = ["Timecode (sec)", "Coley Prediction (0 no movement, 1 movement)", "AC > 2 prediction (0 no movement, 1 movement)"]\nfor name_nb in len(names_column) :\n    my_sheet.cell(row = 1, column = name_nb)\n    my_sheet.value = names_column[name_nb]\n\n# 1ere colonne = seconde\nfor idx_sec in range(len(list_dom_sec)):\n    my_sheet.cell(row = idx_sec + 1, column = 0)\n    my_sheet.value = list_dom_sec[idx_sec]\n\n# 2e colonne = résultats Coley\n\n# 3e colonne = résultats AC > 2'